# Experiment 2 — PaliGemma v2 Location State Extraction

Evaluates every PaliGemma v2 checkpoint listed in `models.json` on location
extraction from indoor campus signage. Two categories:

| Category | Images | Questions | Purpose |
|----------|--------|-----------|---------|
| `building_sign` | 12 | 1 combined (building code + floor number) | Building-level identification |
| `door_room` | 10 | 3 (detection, room code, door state) | Room-level identification |

## Metrics

| Metric | Definition |
|--------|------------|
| Building accuracy | Correct building code extracted (token-based match over the accepted variants) |
| Floor accuracy | Correct floor number extracted (regex over `{-2, -1, 0, 1, 2}`) |
| Location accuracy | Building AND floor correct, complete images only |
| Hallucination rate | Fraction of non-building images where the model invents a building code |
| Room code accuracy | Expected room code appears in the prediction (word-boundary, negation-guarded) |
| BERTScore F1 | Semantic similarity for complete building signs |
| VRAM peak (GB) | Max GPU memory per model |
| Avg inference time (ms) | Mean latency per call over `N_LATENCY_RUNS` passes |

## Image types (building_sign)

| Type | Images | Building expected? |
|------|--------|--------------------|
| `complete` | 01–06 | Yes (EPS / G) |
| `sign_only` | 07–09 | No (floor only) |
| `wall_number` | 10–12 | No (painted number) |

## Output layout

```
/content/results/exp2/
├── exp2_reproducibility.json
├── <model_safe>/
│   ├── results.csv
│   ├── <model_safe>_exp2_results.csv
│   ├── <model_safe>_exp2_aggregate.csv
│   ├── <model_safe>_exp2_efficiency.json
│   ├── <model_safe>_exp2_matrices.json
│   └── plots/
├── all_models_summary.csv
├── bootstrap_cis.csv
├── statistical_tests.csv
├── comparison_plots/
└── failed_models.json  (written only if a run crashed)
```

## Before running (Colab)

1. **Runtime → GPU** — T4 works for 3B; 10B needs an A100 or equivalent.
2. Upload to `/content/`: `images.zip`, `annotations.zip`, `models.json`.
3. Set `HF_TOKEN` in §3.
4. Run all cells.


## 1. Install dependencies

Colab already ships with a CUDA-enabled torch build, so torch is intentionally
not reinstalled here (pip would replace it with the CPU-only wheel and break
the GPU path).


In [ ]:
!pip install -q transformers accelerate bert-score Pillow scipy seaborn


## 2. Unzip dataset

Expects `images.zip`, `annotations.zip` and `models.json` to be in `/content/`.
Both zips are idempotent: the extractor skips work if the target folder already
exists. Annotations may ship either flat or under an `annotations/` subfolder;
the path resolver handles both.


In [ ]:
import os
import zipfile


def unzip_if_needed(zip_path, extract_to):
    if os.path.exists(zip_path) and not os.path.exists(extract_to):
        print(f'Unzipping {zip_path} -> {extract_to}')
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(extract_to)
        print('Done.')
    elif os.path.exists(extract_to):
        print(f'Already extracted: {extract_to}')
    else:
        print(f'ZIP not found: {zip_path}')


unzip_if_needed('/content/images.zip', '/content/images')
unzip_if_needed('/content/annotations.zip', '/content/annotations')


## 3. Configuration

Paths, decoding protocol, seeds and the model list. All 8 PaliGemma v2
checkpoints are evaluated in sequence; the loop is full-batch so no subset
filter is exposed.


In [ ]:
import json
import random
import numpy as np
import torch

HF_TOKEN = ''  # Set your HuggingFace token here (needed for gated PaliGemma weights).

IMAGES_BASE      = '/content/images'
ANNOTATIONS_BASE = '/content/annotations'
MODELS_JSON_PATH = '/content/models.json'
RESULTS_PATH     = '/content/results/exp2'
MAX_NEW_TOKENS   = 30


def _resolve_ann_path(base):
    """Return the folder that actually contains questions.json."""
    if os.path.isfile(os.path.join(base, 'questions.json')):
        return base
    nested = os.path.join(base, 'annotations')
    if os.path.isfile(os.path.join(nested, 'questions.json')):
        print(f'Note: annotations inside subfolder, using {nested}')
        return nested
    return base


def _resolve_images_path(base):
    """Return the folder that contains the per-category image subfolders."""
    def has_exp2(root):
        return (os.path.isdir(os.path.join(root, 'building_sign'))
                or os.path.isdir(os.path.join(root, 'door_room')))
    if has_exp2(base):
        return base
    nested = os.path.join(base, 'images')
    if has_exp2(nested):
        print(f'Note: image categories inside subfolder, using {nested}')
        return nested
    return base


IMAGES_PATH      = _resolve_images_path(IMAGES_BASE)
ANNOTATIONS_PATH = _resolve_ann_path(ANNOTATIONS_BASE)

with open(MODELS_JSON_PATH) as f:
    MODELS = json.load(f)['models']
print(f'Loaded {len(MODELS)} models from models.json')

os.makedirs(RESULTS_PATH, exist_ok=True)
os.makedirs(f'{RESULTS_PATH}/comparison_plots', exist_ok=True)

# Deterministic seeds.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DECODING_PROTOCOL = {
    'do_sample': False,
    'num_beams': 1,
    'temperature': None,
    'max_new_tokens': MAX_NEW_TOKENS,
    'torch_dtype': 'bfloat16 on CUDA, float32 on CPU',
}

# Latency benchmarking:
#   - N_WARMUP synthetic inferences per model are discarded before timing.
#   - Each model is then timed N_LATENCY_RUNS times over all tasks.
#   - inference_time_ms = mean across the N_LATENCY_RUNS per-task values.
#   - Predictions are collected on run 0 only because greedy decoding with
#     num_beams=1 is deterministic and the extra runs add latency signal only.
N_WARMUP = 3
N_LATENCY_RUNS = 3

print(f'Seed: {SEED} | N_WARMUP: {N_WARMUP} | N_LATENCY_RUNS: {N_LATENCY_RUNS}')
print(f'Images:      {IMAGES_PATH}')
print(f'Annotations: {ANNOTATIONS_PATH}')
print(f'Results:     {RESULTS_PATH}')


## 4. Reproducibility card

Captures GPU, library versions, dataset SHA-256 hashes, the decoding protocol
and the latency protocol. Written to `exp2_reproducibility.json` so the run is
reproducible exactly as documented.


In [ ]:
import subprocess
import datetime
import platform
import hashlib
import transformers
from importlib.metadata import version as _pkg_version_


def get_git_commit():
    try:
        return subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'],
                                       stderr=subprocess.DEVNULL).decode().strip()
    except Exception:
        return 'N/A'


def _pkg_version(name):
    try:
        return _pkg_version_(name)
    except Exception:
        return 'N/A'


def _file_sha256(path, chunk_size=65536):
    """SHA-256 of a file, streamed in chunks so large zips do not spike RAM."""
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()


_artifact_candidates = [
    '/content/images.zip',
    '/content/annotations.zip',
    os.path.join(ANNOTATIONS_PATH, 'exp2_expected_answers.json'),
    os.path.join(ANNOTATIONS_PATH, 'questions.json'),
    '/content/models.json',
]
dataset_artifact_hashes = {}
for p in _artifact_candidates:
    if os.path.exists(p):
        dataset_artifact_hashes[p] = {
            'sha256': _file_sha256(p),
            'size_bytes': os.path.getsize(p),
        }

LATENCY_PROTOCOL = {
    'n_warmup': N_WARMUP,
    'n_latency_runs': N_LATENCY_RUNS,
    'cuda_synchronize_around_timer': True,
    'warmup_image_source': 'synthetic 224x224 deterministic RGB noise, seed=SEED',
    'image_order': 'deterministically shuffled per latency run (seed = SEED*1000 + run_idx)',
}

repro = {
    'experiment':            'exp2',
    'date':                  datetime.datetime.now().isoformat(),
    'python_version':        platform.python_version(),
    'torch_version':         torch.__version__,
    'transformers_version':  transformers.__version__,
    'pillow_version':        _pkg_version('pillow'),
    'bert_score_version':    _pkg_version('bert-score'),
    'cuda_available':        torch.cuda.is_available(),
    'gpu_name':              torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
    'git_commit':            get_git_commit(),
    'seed':                  SEED,
    'cudnn_deterministic':   True,
    'models_evaluated':      [m['model_id'] for m in MODELS],
    'max_new_tokens':        MAX_NEW_TOKENS,
    'decoding_protocol':     DECODING_PROTOCOL,
    'latency_protocol':      LATENCY_PROTOCOL,
    'dataset_artifact_hashes': dataset_artifact_hashes,
}

repro_path = f'{RESULTS_PATH}/exp2_reproducibility.json'
with open(repro_path, 'w') as f:
    json.dump(repro, f, indent=2)
print(f'Reproducibility card saved to {repro_path}')

preview = {k: v for k, v in repro.items() if k != 'dataset_artifact_hashes'}
preview['dataset_artifact_hashes'] = f'<{len(dataset_artifact_hashes)} SHA-256 fingerprints saved to JSON>'
print(json.dumps(preview, indent=2))


## 5. Imports and dataset loading

Loads `questions.json` and `exp2_expected_answers.json`, and caches the
building and door-room question definitions for later cells.


In [ ]:
import time
import gc
import re
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from bert_score import score as bertscore_fn
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

with open(f'{ANNOTATIONS_PATH}/questions.json') as f:
    questions_data = json.load(f)
with open(f'{ANNOTATIONS_PATH}/exp2_expected_answers.json') as f:
    expected_answers = json.load(f)

bs_questions = questions_data['experiment2']['categories']['building_sign']['questions']
dr_questions = questions_data['experiment2']['categories']['door_room']['questions']

print('Annotations loaded.')
print(f'  building_sign: {len(expected_answers.get("building_sign", {}))} images')
print(f'  door_room:     {len(expected_answers.get("door_room", {}))} images')


## 6. Metric primitives

Five dedicated metrics:

- `calc_building_accuracy` — token-based match so single-letter codes like `G`
  never match incidentally inside common words.
- `calc_floor_accuracy` — regex over the valid floor set `{-2, -1, 0, 1, 2}`.
- `calc_location_accuracy` — both building and floor correct.
- `calc_room_code_accuracy` — word-boundary match guarded against negations.
- `calc_exact_match` — the yes/no scorer shared with Exp1.

`is_building_hallucinated` flags the model inventing a building code on
`sign_only` / `wall_number` images where no building is visible.


In [ ]:
VALID_FLOORS = {'-2', '-1', '0', '1', '2'}

_NEGATION_REGEX = (
    r"\bno\b|\bnot\b|\bnone\b|\bnever\b|\bcannot\b|\bcan't\b"
    r"|\bthere (?:is|are) no\b"
    r"|\bi (?:do not|don't|cannot|can't) see\b"
    r"|\bnot (?:visible|present)\b"
)


def _load_building_codes():
    """Set of campus building codes (lowercase) for hallucination detection."""
    candidates = [
        os.path.join(ANNOTATIONS_PATH, 'campus_topology.json'),
        '/content/annotations/campus_topology.json',
        '/content/campus_topology.json',
    ]
    for p in candidates:
        if os.path.exists(p):
            try:
                with open(p, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                codes = set()
                for b in data.get('buildings', []):
                    if b.get('name'):
                        codes.add(b['name'].lower())
                    for alias in b.get('aliases', []):
                        codes.add(str(alias).lower())
                if codes:
                    return codes
            except Exception:
                continue
    # Defensive fallback: 16 known UFV campus building codes.
    return {'g', 'cd', 'h', 'p', 'o', 'm', 'n', 'csq', 'k', 'csc',
            'c', 'e', 'cm', 'lcb', 'rodilla', 'capilla', 'eps'}


_BUILDING_CODES = _load_building_codes()
print(f'Building codes loaded for hallucination check: {sorted(_BUILDING_CODES)}')


def normalize(text):
    return str(text).lower().strip()


def extract_floor_from_response(prediction):
    pred = normalize(prediction)
    for token in pred.split():
        if token in VALID_FLOORS:
            return token
    for m in re.finditer(r'-?\d+', pred):
        if m.group() in VALID_FLOORS:
            return m.group()
    return None


def calc_building_accuracy(prediction, expected_building, building_acceptable):
    if expected_building is None:
        return None
    pred = normalize(prediction)
    pred_tokens = set(re.findall(r"[a-z]+|-?\d+", pred))
    for variant in (building_acceptable or []):
        variant_words = normalize(str(variant)).split()
        if len(variant_words) == 1:
            if variant_words[0] in pred_tokens:
                return 1.0
        elif all(w in pred_tokens for w in variant_words):
            return 1.0
    return 0.0


def calc_floor_accuracy(prediction, expected_floor):
    if expected_floor is None:
        return None
    extracted = extract_floor_from_response(prediction)
    return 1.0 if extracted == str(expected_floor) else 0.0


def calc_location_accuracy(building_ok, floor_ok):
    if building_ok is None or floor_ok is None:
        return None
    return 1.0 if (building_ok == 1.0 and floor_ok == 1.0) else 0.0


def calc_room_code_accuracy(prediction, expected_code, room_code_acceptable):
    pred = normalize(prediction)
    if re.search(_NEGATION_REGEX, pred):
        return 0.0

    def wb_hit(needle):
        if not needle:
            return False
        return bool(re.search(r"\b" + re.escape(needle) + r"\b", pred))

    for variant in (room_code_acceptable or []):
        if wb_hit(normalize(str(variant))):
            return 1.0
    return 1.0 if wb_hit(normalize(str(expected_code))) else 0.0


def calc_exact_match(prediction, expected):
    """Robust exact match for closed-vocabulary and yes/no questions."""
    pred = normalize(prediction)
    exp = normalize(expected)
    if not pred or not exp:
        return 0.0
    if exp in ('yes', 'no'):
        is_neg = bool(re.search(_NEGATION_REGEX, pred))
        is_pos = bool(re.search(
            r"\byes\b|\byeah\b|\byep\b"
            r"|\bi (?:can see|see)\b"
            r"|\bthere (?:is|are) (?:a |an |some )?\w"
            r"|\b(?:visible|present)\b"
            r"|\bit is\b|\bthis is\b|\bthat is\b",
            pred,
        )) and not is_neg
        if exp == 'yes':
            return 1.0 if is_pos else 0.0
        return 1.0 if (is_neg and not is_pos) else 0.0
    tokens = pred.split()
    if tokens and tokens[0] == exp:
        return 1.0
    return 1.0 if re.search(r"\b" + re.escape(exp) + r"\b", pred) else 0.0


def is_building_hallucinated(prediction):
    """True if the prediction contains a UFV building code as a whole word."""
    pred = normalize(prediction)
    tokens = re.findall(r"\b[a-z]+\b", pred)
    return any(t in _BUILDING_CODES for t in tokens)


print('Metric primitives defined')


## 7. Per-model plot helpers

Each model produces five plots: floor-confusion heatmap, door-state heatmap,
per-sign-type floor accuracy, room-code accuracy per image, and an overall
metrics bar chart.


In [ ]:
def save_per_model_plots(results_df, model_dir, model_safe):
    plots_dir = f'{model_dir}/plots'
    os.makedirs(plots_dir, exist_ok=True)

    # Floor confusion heatmap
    bs_df = results_df[
        (results_df['category'] == 'building_sign') &
        (results_df['question_id'] == 'building_sign_combined')
    ].copy()
    floor_labels = ['-2', '-1', '0', '1', '2', 'other']
    fl_mat = np.zeros((6, 6), dtype=int)
    fl_idx = {l: i for i, l in enumerate(floor_labels)}
    for _, row in bs_df.iterrows():
        if row['expected_floor'] is None:
            continue
        gt = str(row['expected_floor']) if str(row['expected_floor']) in fl_idx else 'other'
        pred = extract_floor_from_response(str(row['prediction'])) or 'other'
        fl_mat[fl_idx[gt], fl_idx[pred]] += 1
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(fl_mat, annot=True, fmt='d', xticklabels=floor_labels,
                yticklabels=floor_labels, cmap='Blues', ax=ax)
    ax.set_xlabel('Predicted Floor')
    ax.set_ylabel('Expected Floor')
    ax.set_title(f'Floor Confusion\n{model_safe}')
    plt.tight_layout()
    plt.savefig(f'{plots_dir}/{model_safe}_floor_confusion.png', dpi=150)
    plt.close()

    # Door state heatmap
    ds_df = results_df[results_df['question_id'] == 'doorroom_state'].copy()
    ds_labels = ['open', 'closed']
    ds_mat = np.zeros((2, 2), dtype=int)
    ds_idx = {l: i for i, l in enumerate(ds_labels)}
    for _, row in ds_df.iterrows():
        gt = normalize(str(row.get('expected_value', '')))
        pred_tok = normalize(str(row['prediction'])).split()
        pred = pred_tok[0] if pred_tok else ''
        if gt in ds_idx and pred in ds_idx:
            ds_mat[ds_idx[gt], ds_idx[pred]] += 1
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(ds_mat, annot=True, fmt='d', xticklabels=ds_labels,
                yticklabels=ds_labels, cmap='Blues', ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Ground Truth')
    ax.set_title(f'Door State\n{model_safe}')
    plt.tight_layout()
    plt.savefig(f'{plots_dir}/{model_safe}_door_state_confusion.png', dpi=150)
    plt.close()

    # Floor accuracy by sign type
    type_acc = {}
    for itype in ['complete', 'sign_only', 'wall_number']:
        sub = bs_df[bs_df['image_type'] == itype]
        type_acc[itype] = sub['floor_accuracy'].dropna().mean() if len(sub) else None
    types = [t for t, v in type_acc.items() if v is not None]
    vals = [type_acc[t] for t in types]
    colors = ['#4CAF50' if v >= 0.9 else '#FF9800' if v >= 0.7 else '#F44336' for v in vals]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(types, vals, color=colors)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Floor Accuracy')
    ax.set_title(f'Floor Accuracy by Sign Type\n{model_safe}')
    for i, v in enumerate(vals):
        ax.text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=10)
    plt.tight_layout()
    plt.savefig(f'{plots_dir}/{model_safe}_type_breakdown.png', dpi=150)
    plt.close()

    # Room code accuracy per image
    rc_df = results_df[results_df['question_id'] == 'doorroom_code'].copy()
    if len(rc_df) > 0:
        rc_by_image = rc_df.groupby('image')['room_code_accuracy'].mean().reset_index()
        fig, ax = plt.subplots(figsize=(8, 4))
        colors = ['#4CAF50' if v >= 0.9 else '#F44336' for v in rc_by_image['room_code_accuracy']]
        ax.bar(rc_by_image['image'], rc_by_image['room_code_accuracy'], color=colors)
        ax.set_ylim(0, 1.2)
        ax.set_ylabel('Room Code Accuracy')
        ax.set_title(f'Room Code Accuracy per Image\n{model_safe}')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.savefig(f'{plots_dir}/{model_safe}_room_code_accuracy.png', dpi=150)
        plt.close()

    # Overall metrics
    bs_comb = results_df[results_df['question_id'] == 'building_sign_combined']
    metrics = {
        'Building Acc':   bs_comb['building_accuracy'].dropna().mean(),
        'Floor Acc':      bs_comb['floor_accuracy'].dropna().mean(),
        'Location Acc':   bs_comb['location_accuracy'].dropna().mean(),
        'Room Code Acc':  results_df[results_df['question_id'] == 'doorroom_code']['room_code_accuracy'].dropna().mean(),
        'Detection Acc':  results_df[results_df['question_id'] == 'doorroom_detection']['exact_match'].dropna().mean(),
        'Door State Acc': results_df[results_df['question_id'] == 'doorroom_state']['exact_match'].dropna().mean(),
    }
    colors = ['#4CAF50' if v >= 0.9 else '#FF9800' if v >= 0.7 else '#F44336'
              for v in metrics.values() if v is not None]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(list(metrics.keys()), list(metrics.values()), color=colors)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel('Score')
    ax.set_title(f'Overall Metrics - {model_safe}')
    for i, v in enumerate(metrics.values()):
        if v is not None and not np.isnan(v):
            ax.text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=9)
    plt.xticks(rotation=20, ha='right')
    plt.tight_layout()
    plt.savefig(f'{plots_dir}/{model_safe}_accuracy_by_category.png', dpi=150)
    plt.close()


print('Plot helpers defined')


## 8. Model loading

Loads the PaliGemma processor and weights. `make_synthetic_image` returns a
deterministic RGB-noise image used for warmup, disjoint from the measured set.


In [ ]:
def make_synthetic_image(seed=None):
    """Deterministic 224x224 RGB-noise PIL image used for model warmup."""
    rng = np.random.default_rng(SEED if seed is None else seed)
    arr = rng.integers(0, 256, size=(224, 224, 3), dtype=np.uint8)
    return Image.fromarray(arr, mode='RGB')


def load_paligemma(meta):
    """Load PaliGemma model + processor on the active device. Returns (model, processor)."""
    model_id = meta['model_id']
    dtype = torch.bfloat16 if DEVICE == 'cuda' else torch.float32
    model = PaliGemmaForConditionalGeneration.from_pretrained(
        model_id, token=HF_TOKEN, torch_dtype=dtype,
    ).to(DEVICE).eval()
    processor = AutoProcessor.from_pretrained(model_id, token=HF_TOKEN)
    return model, processor


print('Model loader defined')


## 9. Inference primitives

Standard PaliGemma flow: `<image>{question}` → tokenise → generate → decode.
`run_inference_timed` synchronises around the timer so GPU kernel enqueue
latency is not counted as inference work.


In [ ]:
def build_paligemma_inputs(processor, image, question, model):
    prompt = f'<image>{question}'
    inputs = processor(text=prompt, images=image,
                       padding='longest', do_convert_rgb=True,
                       return_tensors='pt').to(DEVICE)
    return inputs.to(dtype=model.dtype)


def decode_paligemma(processor, inputs, outputs):
    input_len = inputs['input_ids'].shape[-1]
    return processor.decode(outputs[0][input_len:], skip_special_tokens=True).strip()


def _sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def run_inference_timed(processor, model, image, question):
    """Generate one answer and return (prediction, latency_ms)."""
    inputs = build_paligemma_inputs(processor, image, question, model)
    _sync()
    t0 = time.time()
    gen_kwargs = dict(
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        num_beams=1,
        pad_token_id=processor.tokenizer.pad_token_id or processor.tokenizer.eos_token_id,
        eos_token_id=processor.tokenizer.eos_token_id,
    )
    with torch.no_grad():
        output = model.generate(**inputs, **gen_kwargs)
    _sync()
    latency_ms = (time.time() - t0) * 1000.0
    text = decode_paligemma(processor, inputs, output)
    return text, latency_ms


print('Inference primitives defined')


## 10. Task list builder

Flattens every `(image, question)` into a stable task list so the
`N_LATENCY_RUNS` passes can shuffle the order without touching metric logic.
Images are decoded once and cached.


In [ ]:
def build_task_list(images_path, bs_questions, dr_questions, expected_answers):
    tasks = []
    q_combined = next(q for q in bs_questions if q['id'] == 'building_sign_combined')

    bs_ans = expected_answers.get('building_sign', {})
    for img_name, ann in bs_ans.items():
        img_path = f'{images_path}/building_sign/{img_name}'
        if not os.path.exists(img_path):
            print(f'  WARNING: image not found: {img_path}')
            continue
        tasks.append({
            'category': 'building_sign',
            'image_name': img_name, 'image_path': img_path,
            'image_type': ann.get('type', 'complete'),
            'question_id': 'building_sign_combined',
            'question_text': q_combined['question'],
            'expected_building': ann.get('building'),
            'expected_floor': ann.get('floor'),
            'building_acceptable': ann.get('building_acceptable', []),
            'bertscore_reference': ann.get('bertscore_reference'),
            'expected_value': ann.get('bertscore_reference'),
            'room_code_acceptable': None,
        })

    dr_ans = expected_answers.get('door_room', {})
    for img_name, ann in dr_ans.items():
        img_path = f'{images_path}/door_room/{img_name}'
        if not os.path.exists(img_path):
            print(f'  WARNING: image not found: {img_path}')
            continue
        rc_acceptable = ann.get('room_code_acceptable', [])
        for q in dr_questions:
            expected_val = ann.get(q.get('expected_answer_key'), '')
            tasks.append({
                'category': 'door_room',
                'image_name': img_name, 'image_path': img_path,
                'image_type': None,
                'question_id': q['id'], 'question_text': q['question'],
                'expected_building': None, 'expected_floor': None,
                'building_acceptable': None, 'bertscore_reference': None,
                'expected_value': expected_val,
                'room_code_acceptable': rc_acceptable,
            })
    return tasks


def preload_images(tasks):
    cache = {}
    for t in tasks:
        if t['image_path'] not in cache:
            cache[t['image_path']] = Image.open(t['image_path']).convert('RGB')
    return cache


print('Task list builder defined')


## 11. Per-model evaluation pipeline

The pipeline is split into four helpers plus an orchestrator. Each helper owns
one phase: row assembly, aggregation, confusion matrices, and the summary row
for the cross-model table.


### 11.1 Assemble result rows

Wraps every `(task, prediction, latencies)` triple into a DataFrame row with
the metric values relevant to the question type.


In [ ]:
def assemble_result_rows(tasks, predictions, latencies, model_id, family):
    rows = []
    for idx, t in enumerate(tasks):
        pred = predictions[idx] or ''
        lats = latencies[idx]
        lat_mean = float(np.mean(lats)) if lats else 0.0
        lat_std = float(np.std(lats, ddof=1)) if len(lats) >= 2 else 0.0

        base = {
            'model_id': model_id, 'family': family, 'category': t['category'],
            'image': t['image_name'], 'image_type': t['image_type'],
            'question_id': t['question_id'], 'question': t['question_text'],
            'prediction': pred,
            'expected_value': t['expected_value'],
            'expected_building': t['expected_building'],
            'expected_floor': t['expected_floor'],
            'bertscore_reference': t['bertscore_reference'],
            'inference_time_ms': round(lat_mean, 2),
            'latency_ms_mean': round(lat_mean, 2),
            'latency_ms_std': round(lat_std, 2),
            'output_length_words': len(pred.split()),
        }
        for i, lat in enumerate(lats, start=1):
            base[f'latency_ms_run{i}'] = round(float(lat), 2)

        if t['category'] == 'building_sign':
            b = calc_building_accuracy(pred, t['expected_building'], t['building_acceptable'])
            f_ = calc_floor_accuracy(pred, t['expected_floor'])
            loc = calc_location_accuracy(b, f_)
            halluc = is_building_hallucinated(pred) if t['expected_building'] is None else None
            base.update({
                'building_accuracy': b, 'floor_accuracy': f_,
                'location_accuracy': loc, 'hallucination': halluc,
                'room_code_accuracy': None, 'exact_match': None,
                'bertscore_f1': None, 'bertscore_p': None, 'bertscore_r': None,
            })
        else:
            rc, em = None, None
            if t['question_id'] == 'doorroom_code':
                rc = calc_room_code_accuracy(pred, t['expected_value'], t['room_code_acceptable'])
            else:
                em = calc_exact_match(pred, str(t['expected_value']))
            base.update({
                'building_accuracy': None, 'floor_accuracy': None,
                'location_accuracy': None, 'hallucination': None,
                'room_code_accuracy': rc, 'exact_match': em,
                'bertscore_f1': None, 'bertscore_p': None, 'bertscore_r': None,
            })
        rows.append(base)
    return pd.DataFrame(rows)


def add_bertscore(df):
    """Fill bertscore_{f1,p,r} for complete building signs only."""
    mask = (df['question_id'] == 'building_sign_combined') & df['bertscore_reference'].notna()
    if mask.sum() == 0:
        return
    preds = df[mask]['prediction'].tolist()
    refs = df[mask]['bertscore_reference'].tolist()
    P, R, F1 = bertscore_fn(preds, refs, lang='en', verbose=False)
    df.loc[mask, 'bertscore_f1'] = [round(x, 4) for x in F1.tolist()]
    df.loc[mask, 'bertscore_p']  = [round(x, 4) for x in P.tolist()]
    df.loc[mask, 'bertscore_r']  = [round(x, 4) for x in R.tolist()]
    print(f'  BERTScore computed on {mask.sum()} complete signs. Mean F1={F1.mean():.4f}')


print('Row assembly helpers defined')


### 11.2 Aggregate per category

One row per category with the category-specific aggregate metrics (building
accuracy, per-type floor accuracy, hallucination rate, room-code accuracy,
door-state accuracy).


In [ ]:
def aggregate_per_category(results_df):
    def mean_or_none(series):
        s = series.dropna()
        return round(float(s.mean()), 4) if len(s) > 0 else None

    def type_floor_acc(df, itype):
        sub = df[df['image_type'] == itype]
        return mean_or_none(sub['floor_accuracy'])

    bs_df = results_df[results_df['category'] == 'building_sign']
    dr_df = results_df[results_df['category'] == 'door_room']

    halluc_series = bs_df[bs_df['expected_building'].isna()]['hallucination']
    halluc_rate = mean_or_none(
        halluc_series.map(lambda x: 1.0 if x else 0.0 if x is not None else None)
    )

    agg_rows = [
        {
            'category': 'building_sign',
            'building_accuracy':          mean_or_none(bs_df['building_accuracy']),
            'floor_accuracy':             mean_or_none(bs_df['floor_accuracy']),
            'location_accuracy':          mean_or_none(bs_df['location_accuracy']),
            'bertscore_f1':               mean_or_none(bs_df['bertscore_f1']),
            'bertscore_p':                mean_or_none(bs_df['bertscore_p']),
            'bertscore_r':                mean_or_none(bs_df['bertscore_r']),
            'complete_building_accuracy': mean_or_none(bs_df[bs_df['image_type'] == 'complete']['building_accuracy']),
            'complete_floor_accuracy':    type_floor_acc(bs_df, 'complete'),
            'complete_location_accuracy': mean_or_none(bs_df[bs_df['image_type'] == 'complete']['location_accuracy']),
            'sign_only_floor_accuracy':   type_floor_acc(bs_df, 'sign_only'),
            'wall_number_floor_accuracy': type_floor_acc(bs_df, 'wall_number'),
            'hallucination_rate':         halluc_rate,
            'avg_output_length':          mean_or_none(bs_df['output_length_words']),
            'doorroom_detection_accuracy': None,
            'doorroom_state_accuracy':    None,
            'room_code_accuracy':         None,
        },
        {
            'category': 'door_room',
            'building_accuracy': None, 'floor_accuracy': None, 'location_accuracy': None,
            'bertscore_f1': None, 'bertscore_p': None, 'bertscore_r': None,
            'complete_building_accuracy': None, 'complete_floor_accuracy': None,
            'complete_location_accuracy': None, 'sign_only_floor_accuracy': None,
            'wall_number_floor_accuracy': None, 'hallucination_rate': None,
            'avg_output_length': mean_or_none(dr_df['output_length_words']),
            'doorroom_detection_accuracy': mean_or_none(dr_df[dr_df['question_id'] == 'doorroom_detection']['exact_match']),
            'doorroom_state_accuracy':     mean_or_none(dr_df[dr_df['question_id'] == 'doorroom_state']['exact_match']),
            'room_code_accuracy':          mean_or_none(dr_df[dr_df['question_id'] == 'doorroom_code']['room_code_accuracy']),
        },
    ]
    return pd.DataFrame(agg_rows), halluc_rate


print('Category aggregation defined')


### 11.3 Confusion matrices

Two matrices per model: a 6×6 floor confusion matrix over
`{-2, -1, 0, 1, 2, other}` and a 2×2 door-state confusion matrix over
`{open, closed}`. Packaged with a hallucination summary.


In [ ]:
def build_confusion_matrices(results_df, halluc_rate):
    bs_comb = results_df[
        (results_df['category'] == 'building_sign') &
        (results_df['question_id'] == 'building_sign_combined')
    ]
    floor_labels = ['-2', '-1', '0', '1', '2', 'other']
    fl_idx = {l: i for i, l in enumerate(floor_labels)}
    fl_mat = np.zeros((6, 6), dtype=int)
    for _, row in bs_comb.iterrows():
        if row['expected_floor'] is None:
            continue
        gt = str(row['expected_floor']) if str(row['expected_floor']) in fl_idx else 'other'
        pred = extract_floor_from_response(str(row['prediction'])) or 'other'
        fl_mat[fl_idx[gt], fl_idx[pred]] += 1
    fl_total = fl_mat.sum()
    fl_acc = round(np.trace(fl_mat) / fl_total, 4) if fl_total > 0 else 0.0

    dr_state = results_df[results_df['question_id'] == 'doorroom_state']
    ds_labels = ['open', 'closed']
    ds_idx = {l: i for i, l in enumerate(ds_labels)}
    ds_mat = np.zeros((2, 2), dtype=int)
    unknown_ds = 0
    for _, row in dr_state.iterrows():
        gt = normalize(str(row.get('expected_value', '')))
        pred_tok = normalize(str(row['prediction'])).split()
        pred = pred_tok[0] if pred_tok else ''
        if gt in ds_idx and pred in ds_idx:
            ds_mat[ds_idx[gt], ds_idx[pred]] += 1
        elif gt in ds_idx:
            unknown_ds += 1
    ds_total = ds_mat.sum()
    ds_acc = round(np.trace(ds_mat) / ds_total, 4) if ds_total > 0 else 0.0

    halluc_series = bs_comb[bs_comb['expected_building'].isna()]['hallucination']
    matrices = {
        'floor_confusion': {
            'labels': floor_labels, 'matrix': fl_mat.tolist(),
            'accuracy': fl_acc, 'total': int(fl_total),
            'note': 'Rows=expected floor, columns=predicted floor',
        },
        'door_state_confusion': {
            'labels': ds_labels, 'matrix': ds_mat.tolist(),
            'accuracy': ds_acc, 'total': int(ds_total),
            'unknown_predictions': unknown_ds,
            'note': 'Rows=ground truth, columns=predicted',
        },
        'hallucination': {
            'rate': halluc_rate,
            'n_sign_only_wall_number': int(halluc_series.notna().sum()),
            'note': 'Fraction of non-building images where model mentioned a building code',
        },
    }
    return matrices, fl_acc, ds_acc


print('Confusion matrices helper defined')


### 11.4 Orchestrator

Composes the helpers: load → warmup → timed inference → row assembly →
BERTScore → aggregation → efficiency → matrices → save → return summary.


In [ ]:
def run_experiment2_for_model(model_meta):
    """Run full Exp2 for one model. Saves per-model files and returns a summary dict."""
    model_id = model_meta['model_id']
    family = model_meta.get('family', 'paligemma')
    model_safe = model_id.replace('/', '_').replace('-', '_')
    model_dir = f'{RESULTS_PATH}/{model_safe}'
    os.makedirs(model_dir, exist_ok=True)

    print(f'\n{"=" * 60}')
    print(f'MODEL: {model_id}')
    print(f'{"=" * 60}')

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    t_load = time.time()
    model, processor = load_paligemma(model_meta)
    model_load_time_s = round(time.time() - t_load, 2)
    n_parameters = sum(p.numel() for p in model.parameters())
    print(f'  Loaded on {DEVICE} -- {n_parameters / 1e9:.2f}B params in {model_load_time_s:.1f}s')

    print(f'  Warming up ({N_WARMUP} iters on synthetic image, discarded)...')
    warmup_img = make_synthetic_image()
    for _ in range(N_WARMUP):
        run_inference_timed(processor, model, warmup_img, 'What is in this image?')

    tasks = build_task_list(IMAGES_PATH, bs_questions, dr_questions, expected_answers)
    images_cache = preload_images(tasks)
    print(f'  {len(tasks)} inference tasks x {N_LATENCY_RUNS} latency runs')

    predictions = [None] * len(tasks)
    per_task_latencies = [[] for _ in tasks]

    for run_idx in range(N_LATENCY_RUNS):
        order = np.arange(len(tasks))
        np.random.default_rng(SEED * 1000 + run_idx).shuffle(order)
        for task_idx in order:
            t = tasks[task_idx]
            image = images_cache[t['image_path']]
            try:
                pred, latency_ms = run_inference_timed(processor, model, image, t['question_text'])
            except Exception as exc:
                print(f'     [FAIL] run {run_idx + 1} task {task_idx} ({t["image_name"]}/{t["question_id"]}): {exc}')
                pred, latency_ms = '', 0.0
            per_task_latencies[task_idx].append(latency_ms)
            if run_idx == 0:
                predictions[task_idx] = pred

    results_df = assemble_result_rows(tasks, predictions, per_task_latencies, model_id, family)
    n_inferences = len(results_df)
    print(f'  {n_inferences} inferences completed (predictions from run 1).')

    add_bertscore(results_df)
    agg_df, halluc_rate = aggregate_per_category(results_df)

    vram_peak = torch.cuda.max_memory_allocated() / (1024 ** 3) if torch.cuda.is_available() else 0.0
    n_images = len(set(results_df['image']))
    total_inf_time = float(results_df['inference_time_ms'].sum())
    avg_inf_ms = total_inf_time / n_inferences if n_inferences > 0 else 0.0

    per_run_means = []
    for run_idx in range(N_LATENCY_RUNS):
        xs = [per_task_latencies[i][run_idx] for i in range(len(tasks))
              if len(per_task_latencies[i]) > run_idx]
        if xs:
            per_run_means.append(float(np.mean(xs)))
    avg_inf_ms_std = float(np.std(per_run_means, ddof=1)) if len(per_run_means) >= 2 else 0.0
    throughput = n_images / (total_inf_time / 1000) if total_inf_time > 0 else 0.0

    efficiency = {
        'model_id': model_id, 'family': family, 'experiment': 'exp2',
        'n_parameters': n_parameters,
        'n_parameters_billions': round(n_parameters / 1e9, 2),
        'n_images': n_images, 'n_inferences': n_inferences,
        'vram_peak_gb': round(vram_peak, 3),
        'model_load_time_s': model_load_time_s,
        'total_inference_time_s': round(total_inf_time / 1000, 3),
        'avg_inference_time_ms': round(avg_inf_ms, 2),
        'avg_inference_time_ms_std': round(avg_inf_ms_std, 2),
        'throughput_img_per_s': round(throughput, 3),
        'throughput_inf_per_s': round(n_inferences / (total_inf_time / 1000), 3) if total_inf_time > 0 else 0,
        'avg_output_length_words': round(results_df['output_length_words'].mean(), 2),
        'max_output_length_words': int(results_df['output_length_words'].max()),
    }

    matrices, fl_acc, ds_acc = build_confusion_matrices(results_df, halluc_rate)
    matrices.update({'model_id': model_id, 'family': family, 'experiment': 'exp2'})

    results_df.to_csv(f'{model_dir}/{model_safe}_exp2_results.csv', index=False)
    results_df.to_csv(f'{model_dir}/results.csv', index=False)
    agg_df.to_csv(f'{model_dir}/{model_safe}_exp2_aggregate.csv', index=False)
    with open(f'{model_dir}/{model_safe}_exp2_efficiency.json', 'w') as f:
        json.dump(efficiency, f, indent=2)
    with open(f'{model_dir}/{model_safe}_exp2_matrices.json', 'w') as f:
        json.dump(matrices, f, indent=2)
    print(f'  Files saved to {model_dir}/')

    save_per_model_plots(results_df, model_dir, model_safe)
    print('  Plots saved.')

    bs_agg = agg_df[agg_df['category'] == 'building_sign'].iloc[0]
    dr_agg = agg_df[agg_df['category'] == 'door_room'].iloc[0]
    summary = {
        'model_id': model_id, 'model_safe': model_safe, 'family': family,
        'version': model_meta.get('version', ''), 'size': model_meta.get('size', ''),
        'type': model_meta.get('type', ''), 'resolution': model_meta.get('resolution', ''),
        'building_accuracy':           bs_agg.get('building_accuracy'),
        'floor_accuracy':              bs_agg.get('floor_accuracy'),
        'location_accuracy':           bs_agg.get('location_accuracy'),
        'bertscore_f1':                bs_agg.get('bertscore_f1'),
        'bertscore_p':                 bs_agg.get('bertscore_p'),
        'bertscore_r':                 bs_agg.get('bertscore_r'),
        'complete_building_accuracy':  bs_agg.get('complete_building_accuracy'),
        'complete_floor_accuracy':     bs_agg.get('complete_floor_accuracy'),
        'complete_location_accuracy':  bs_agg.get('complete_location_accuracy'),
        'sign_only_floor_accuracy':    bs_agg.get('sign_only_floor_accuracy'),
        'wall_number_floor_accuracy':  bs_agg.get('wall_number_floor_accuracy'),
        'hallucination_rate':          bs_agg.get('hallucination_rate'),
        'doorroom_detection_accuracy': dr_agg.get('doorroom_detection_accuracy'),
        'doorroom_state_accuracy':     dr_agg.get('doorroom_state_accuracy'),
        'room_code_accuracy':          dr_agg.get('room_code_accuracy'),
        'floor_matrix_accuracy':       fl_acc,
        'door_state_matrix_accuracy':  ds_acc,
        'n_parameters': n_parameters,
        'n_parameters_billions': round(n_parameters / 1e9, 2),
        'vram_peak_gb':                efficiency['vram_peak_gb'],
        'model_load_time_s':           model_load_time_s,
        'avg_inference_time_ms':       efficiency['avg_inference_time_ms'],
        'avg_inference_time_ms_std':   efficiency['avg_inference_time_ms_std'],
        'throughput_img_per_s':        efficiency['throughput_img_per_s'],
        'throughput_inf_per_s':        efficiency['throughput_inf_per_s'],
        'avg_output_length_words':     efficiency['avg_output_length_words'],
        'n_inferences':                n_inferences,
    }

    del model, processor
    images_cache.clear()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print('  GPU memory freed.')
    return summary


print('Orchestrator defined')


## 12. Main loop

Runs every model in `models.json` sequentially. Failures are collected in
`failed_models` instead of aborting the batch.


In [ ]:
all_summaries = []
failed_models = []

for model_meta in MODELS:
    print(f'\n>>> Running {model_meta["model_id"]}')
    try:
        summary = run_experiment2_for_model(model_meta)
        all_summaries.append(summary)
        print(f'  [OK] {model_meta["model_id"]}')
    except Exception as exc:
        print(f'  [FAIL] {model_meta["model_id"]} -- {exc}')
        failed_models.append({'model_id': model_meta['model_id'], 'error': str(exc)})
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print(f'\n{len(all_summaries)}/{len(MODELS)} models completed successfully.')
if failed_models:
    print(f'Failed: {[m["model_id"] for m in failed_models]}')


## 13. Cross-model comparison table

Key metrics side by side plus a "best per metric" quick view.


In [ ]:
summary_df = pd.DataFrame(all_summaries)

display_cols = [
    'model_id', 'family', 'version', 'size', 'type', 'resolution',
    'n_parameters_billions',
    'building_accuracy', 'floor_accuracy', 'location_accuracy',
    'bertscore_f1', 'hallucination_rate',
    'room_code_accuracy', 'doorroom_detection_accuracy', 'doorroom_state_accuracy',
    'vram_peak_gb', 'model_load_time_s', 'avg_inference_time_ms',
    'throughput_img_per_s', 'throughput_inf_per_s',
]
display_df = summary_df[[c for c in display_cols if c in summary_df.columns]].copy()
float_cols = [c for c in display_df.columns if display_df[c].dtype == float]
display_df[float_cols] = display_df[float_cols].round(4)

print('\n=== EXPERIMENT 2 - ALL MODELS SUMMARY ===')
print(display_df.to_string(index=False))

print('\n=== BEST MODEL PER METRIC ===')
for col in ['building_accuracy', 'floor_accuracy', 'location_accuracy',
            'bertscore_f1', 'room_code_accuracy', 'doorroom_state_accuracy']:
    if col in summary_df.columns:
        s = summary_df[col].dropna()
        if len(s) > 0:
            best_idx = s.idxmax()
            print(f'  {col}: {summary_df.loc[best_idx, "model_id"]} ({s.max():.4f})')

if 'hallucination_rate' in summary_df.columns:
    s = summary_df['hallucination_rate'].dropna()
    if len(s) > 0:
        best_idx = s.idxmin()
        print(f'  lowest hallucination_rate: {summary_df.loc[best_idx, "model_id"]} ({s.min():.4f})')


## 14. Statistical significance

Bootstrap 95% CIs on item-level correctness and pairwise McNemar (exact
binomial on discordant pairs) with Holm–Bonferroni correction. Reads from the
per-model `results.csv` so the cell is self-contained after a crash.


In [ ]:
from scipy.stats import binomtest

N_BOOTSTRAP = 1000


def bootstrap_ci(values, n_boot=N_BOOTSTRAP, ci=95, rng=None):
    if rng is None:
        rng = np.random.default_rng(SEED)
    arr = np.asarray([v for v in values if v is not None and np.isfinite(v)], dtype=float)
    if arr.size == 0:
        return float('nan'), float('nan'), float('nan')
    idx = rng.integers(0, arr.size, size=(n_boot, arr.size))
    samples = arr[idx].mean(axis=1)
    lo = float(np.percentile(samples, (100 - ci) / 2))
    hi = float(np.percentile(samples, 100 - (100 - ci) / 2))
    return float(arr.mean()), lo, hi


def holm_bonferroni(pvals):
    p = np.asarray(pvals, dtype=float)
    n = p.size
    order = np.argsort(p)
    adj = np.empty(n)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (n - rank) * p[i])
        adj[i] = min(running, 1.0)
    return adj


em_by_model = {}
for entry in os.listdir(RESULTS_PATH):
    model_dir = os.path.join(RESULTS_PATH, entry)
    csv_path = os.path.join(model_dir, 'results.csv')
    if not os.path.isfile(csv_path):
        continue
    df = pd.read_csv(csv_path)
    em_map = {}
    for _, row in df.iterrows():
        key = f"{row['image']}::{row['question_id']}"
        if row['category'] == 'building_sign':
            v = row.get('location_accuracy')
        elif row['question_id'] == 'doorroom_code':
            v = row.get('room_code_accuracy')
        else:
            v = row.get('exact_match')
        if pd.isna(v):
            continue
        em_map[key] = float(v)
    em_by_model[entry] = em_map

if not em_by_model:
    print('No per-model results found yet. Run the main loop first.')
    ci_df = pd.DataFrame()
else:
    ci_rows = []
    for model_safe, em_map in em_by_model.items():
        vals = np.array(list(em_map.values()), dtype=float) * 100.0
        mean, lo, hi = bootstrap_ci(vals, rng=np.random.default_rng(SEED))
        ci_rows.append({
            'model_safe': model_safe,
            'n_items': int(vals.size),
            'exact_match_mean_pct': round(mean, 2) if np.isfinite(mean) else None,
            'exact_match_ci_low_pct': round(lo, 2) if np.isfinite(lo) else None,
            'exact_match_ci_high_pct': round(hi, 2) if np.isfinite(hi) else None,
        })
    ci_df = pd.DataFrame(ci_rows).sort_values('exact_match_mean_pct', ascending=False).reset_index(drop=True)
    ci_df.to_csv(f'{RESULTS_PATH}/bootstrap_cis.csv', index=False)
    print(f'Saved: {RESULTS_PATH}/bootstrap_cis.csv ({N_BOOTSTRAP} resamples per row)')

    models_sorted = sorted(em_by_model.keys())
    pairs = []
    raw_p = []
    for i, a in enumerate(models_sorted):
        for b in models_sorted[i + 1:]:
            common_keys = sorted(set(em_by_model[a]) & set(em_by_model[b]))
            if not common_keys:
                continue
            y_a = np.array([em_by_model[a][k] for k in common_keys], dtype=float)
            y_b = np.array([em_by_model[b][k] for k in common_keys], dtype=float)
            b_only = int(np.sum((y_a < 0.5) & (y_b >= 0.5)))
            a_only = int(np.sum((y_a >= 0.5) & (y_b < 0.5)))
            n_disc = a_only + b_only
            if n_disc == 0:
                p_value = 1.0
            else:
                res = binomtest(min(a_only, b_only), n=n_disc, p=0.5, alternative='two-sided')
                p_value = float(res.pvalue)
            mean_a = float(y_a.mean()) * 100.0
            mean_b = float(y_b.mean()) * 100.0
            pairs.append({
                'model_a': a, 'model_b': b,
                'n_common_items': len(common_keys),
                'mean_em_a_pct': round(mean_a, 2),
                'mean_em_b_pct': round(mean_b, 2),
                'better_model': a if mean_a > mean_b else b,
                'p_value_raw': round(p_value, 6),
            })
            raw_p.append(p_value)

    if pairs:
        for row, adj in zip(pairs, holm_bonferroni(raw_p)):
            row['p_value_holm'] = round(float(adj), 6)
            row['significant_0_05'] = bool(adj < 0.05)
        tests_df = pd.DataFrame(pairs).sort_values('p_value_holm').reset_index(drop=True)
        tests_df.to_csv(f'{RESULTS_PATH}/statistical_tests.csv', index=False)
        n_sig = int(tests_df['significant_0_05'].sum())
        print(f'Saved: {RESULTS_PATH}/statistical_tests.csv ({len(tests_df)} pairs, Holm-corrected)')
        print(f'Significant pairs (Holm p<0.05): {n_sig}/{len(tests_df)}')


## 15. CI ranking plot

Horizontal bar chart showing exact-match percentage per model with the 95%
bootstrap CI as error bars.


In [ ]:
if len(ci_df):
    plot_df = ci_df.sort_values('exact_match_mean_pct', ascending=False).reset_index(drop=True)
    means = plot_df['exact_match_mean_pct'].values.astype(float)
    lows = means - plot_df['exact_match_ci_low_pct'].values.astype(float)
    highs = plot_df['exact_match_ci_high_pct'].values.astype(float) - means

    fig, ax = plt.subplots(figsize=(10, max(5, 0.4 * len(plot_df) + 2)))
    y = np.arange(len(plot_df))
    ax.barh(y, means, xerr=[lows, highs], color='#4CAF50', ecolor='black',
            capsize=3, error_kw={'linewidth': 0.8}, alpha=0.85)
    ax.set_yticks(y)
    ax.set_yticklabels(plot_df['model_safe'].values, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel(f'Exact match (%)  --  error bars = 95% bootstrap CI over {N_BOOTSTRAP} resamples')
    ax.set_title('Exact-match ranking with confidence intervals')
    ax.grid(axis='x', alpha=0.3)
    pad = (max(highs) if len(highs) else 1.0) * 0.05
    for i, m in enumerate(means):
        if np.isfinite(m):
            ax.text(m + pad, i, f'{m:.2f}', va='center', fontsize=8)
    plt.tight_layout()
    out = f'{RESULTS_PATH}/comparison_plots/cmp_exact_match_with_cis.png'
    os.makedirs(os.path.dirname(out), exist_ok=True)
    plt.savefig(out, dpi=150)
    plt.close()
    print(f'Saved: {out}')
else:
    print('Skipping CI plot: ci_df is empty.')


## 16. Cross-model plots

One bar chart per headline metric plus VRAM, inference time, and a grouped
chart comparing floor accuracy across the three sign types.


In [ ]:
cmp_dir = f'{RESULTS_PATH}/comparison_plots'
os.makedirs(cmp_dir, exist_ok=True)

summary_df['short_name'] = summary_df['model_id'].apply(
    lambda x: x.split('/')[-1].replace('paligemma', 'pg').replace('paligemma2', 'pg2')
)


def comparison_bar(metric, title, color_thresholds=(0.9, 0.7), lower_is_better=False,
                   filename=None, ylabel='Score'):
    if metric not in summary_df.columns:
        return
    s = summary_df[['short_name', metric]].dropna()
    if len(s) == 0:
        return
    vals = s[metric].values
    if lower_is_better:
        colors = ['#4CAF50' if v <= 0.1 else '#FF9800' if v <= 0.3 else '#F44336' for v in vals]
    else:
        colors = ['#4CAF50' if v >= color_thresholds[0] else '#FF9800' if v >= color_thresholds[1] else '#F44336'
                  for v in vals]
    fig, ax = plt.subplots(figsize=(12, 5))
    bars = ax.bar(s['short_name'], vals, color=colors)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    plt.xticks(rotation=40, ha='right', fontsize=9)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                f'{v:.3f}', ha='center', fontsize=8)
    plt.tight_layout()
    plt.savefig(f'{cmp_dir}/{filename or f"cmp_{metric}.png"}', dpi=150)
    plt.close()


comparison_bar('building_accuracy',           'Building Accuracy -- All Models',           filename='cmp_building_accuracy.png')
comparison_bar('floor_accuracy',              'Floor Accuracy -- All Models',              filename='cmp_floor_accuracy.png')
comparison_bar('location_accuracy',           'Location Accuracy (complete) -- All Models', filename='cmp_location_accuracy.png')
comparison_bar('bertscore_f1',                'BERTScore F1 (complete signs) -- All Models', filename='cmp_bertscore_f1.png')
comparison_bar('hallucination_rate',          'Hallucination Rate -- All Models (lower is better)',
               lower_is_better=True, filename='cmp_hallucination_rate.png', ylabel='Rate')
comparison_bar('room_code_accuracy',          'Room Code Accuracy -- All Models',          filename='cmp_room_code_accuracy.png')
comparison_bar('doorroom_detection_accuracy', 'Room Detection Accuracy -- All Models',     filename='cmp_doorroom_detection.png')
comparison_bar('doorroom_state_accuracy',     'Door State Accuracy -- All Models',         filename='cmp_doorroom_state.png')

if 'vram_peak_gb' in summary_df.columns:
    s = summary_df[['short_name', 'vram_peak_gb']].dropna()
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(s['short_name'], s['vram_peak_gb'], color='#2196F3')
    ax.set_ylabel('VRAM Peak (GB)')
    ax.set_title('GPU VRAM Usage -- All Models')
    plt.xticks(rotation=40, ha='right', fontsize=9)
    for i, v in enumerate(s['vram_peak_gb']):
        ax.text(i, v + 0.1, f'{v:.2f}', ha='center', fontsize=8)
    plt.tight_layout()
    plt.savefig(f'{cmp_dir}/cmp_vram.png', dpi=150)
    plt.close()

if 'avg_inference_time_ms' in summary_df.columns:
    s = summary_df[['short_name', 'avg_inference_time_ms']].dropna()
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(s['short_name'], s['avg_inference_time_ms'], color='#9C27B0')
    ax.set_ylabel('Avg Inference Time (ms)')
    ax.set_title('Average Inference Time -- All Models')
    plt.xticks(rotation=40, ha='right', fontsize=9)
    for i, v in enumerate(s['avg_inference_time_ms']):
        ax.text(i, v + 2, f'{v:.0f}', ha='center', fontsize=8)
    plt.tight_layout()
    plt.savefig(f'{cmp_dir}/cmp_inference_time.png', dpi=150)
    plt.close()

type_cols = ['complete_floor_accuracy', 'sign_only_floor_accuracy', 'wall_number_floor_accuracy']
type_present = [c for c in type_cols if c in summary_df.columns]
if type_present:
    fig, ax = plt.subplots(figsize=(14, 6))
    x = np.arange(len(summary_df))
    w = 0.25
    colors_t = ['#4CAF50', '#2196F3', '#FF9800']
    labels_t = ['Complete', 'Sign Only', 'Wall Number']
    for i, (col, color, label) in enumerate(zip(type_present, colors_t, labels_t)):
        vals = summary_df[col].fillna(0).values
        ax.bar(x + i * w, vals, w, label=label, color=color, alpha=0.85)
    ax.set_xticks(x + w)
    ax.set_xticklabels(summary_df['short_name'], rotation=40, ha='right', fontsize=9)
    ax.set_ylim(0, 1.2)
    ax.set_ylabel('Floor Accuracy')
    ax.set_title('Floor Accuracy by Sign Type -- All Models')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'{cmp_dir}/cmp_floor_by_type.png', dpi=150)
    plt.close()

print(f'All comparison plots saved to {cmp_dir}')


## 17. Save outputs

Final summary CSV, failed-model log, per-metric rankings, and a file inventory.


In [ ]:
summary_path = f'{RESULTS_PATH}/all_models_summary.csv'
summary_df.drop(columns=['short_name'], errors='ignore').to_csv(summary_path, index=False)
print(f'Summary saved: {summary_path}')
print(f'  Shape: {summary_df.shape}')

print('\n=== RANKING: Location Accuracy (complete images) ===')
if 'location_accuracy' in summary_df.columns:
    rank_loc = summary_df[['model_id', 'family', 'version', 'size', 'type', 'resolution',
                           'location_accuracy', 'building_accuracy', 'floor_accuracy',
                           'bertscore_f1']].sort_values('location_accuracy', ascending=False)
    print(rank_loc.to_string(index=False))

print('\n=== RANKING: Room Code Accuracy ===')
if 'room_code_accuracy' in summary_df.columns:
    rank_rc = summary_df[['model_id', 'family', 'version', 'size', 'type', 'resolution',
                          'room_code_accuracy', 'doorroom_detection_accuracy',
                          'doorroom_state_accuracy']].sort_values('room_code_accuracy', ascending=False)
    print(rank_rc.to_string(index=False))

print('\n=== RANKING: VRAM Efficiency ===')
if 'vram_peak_gb' in summary_df.columns:
    eff_cols = ['model_id', 'family', 'size', 'resolution', 'n_parameters_billions',
                'vram_peak_gb', 'model_load_time_s', 'avg_inference_time_ms',
                'throughput_img_per_s', 'throughput_inf_per_s', 'location_accuracy']
    rank_vram = summary_df[[c for c in eff_cols if c in summary_df.columns]].sort_values('vram_peak_gb')
    print(rank_vram.to_string(index=False))

if failed_models:
    with open(f'{RESULTS_PATH}/failed_models.json', 'w') as f:
        json.dump(failed_models, f, indent=2)
    print('\nFailed models log saved.')

print('\n=== FILE INVENTORY ===')
_inventory = [
    'all_models_summary.csv',
    'bootstrap_cis.csv',
    'statistical_tests.csv',
    'exp2_reproducibility.json',
    'comparison_plots/cmp_building_accuracy.png',
    'comparison_plots/cmp_floor_accuracy.png',
    'comparison_plots/cmp_location_accuracy.png',
    'comparison_plots/cmp_room_code_accuracy.png',
    'comparison_plots/cmp_hallucination_rate.png',
    'comparison_plots/cmp_bertscore_f1.png',
    'comparison_plots/cmp_vram.png',
    'comparison_plots/cmp_inference_time.png',
    'comparison_plots/cmp_exact_match_with_cis.png',
]
for rel in _inventory:
    full = os.path.join(RESULTS_PATH, rel)
    tag = '[OK]' if os.path.exists(full) else '[--]'
    print(f'  {tag} {rel}')

print('\n=== EXPERIMENT 2 COMPLETE ===')
print(f'Results directory: {RESULTS_PATH}')
